# Experiment 6: Bagging, Boosting, and Stacked Ensemble Models

```
experiment6_ensemble_breast_cancer.py
========================================
ICS1512 - Machine Learning Algorithms Laboratory
Experiment 6 (Lab Manual "Experiment 7"): Bagging, Boosting, and Stacked
Ensemble Models

Uses the reusable module ml_lab_utils.py (from Experiment 1) for:
    - EDA                         -> generate_eda_summary()
    - Classification train/eval   -> train_evaluate_classification()
    - Classification metrics      -> classification_performance_metrics()
    - Global plot style           -> set_plot_style()

Dataset: Wisconsin Diagnostic Breast Cancer (WDBC), 569 samples, 30 numeric
features, binary target (Malignant / Benign). Loaded from a CSV file
(breast_cancer_data.csv) in the standard Kaggle WDBC format: an "id"
column, a "diagnosis" column ('M' = malignant, 'B' = benign), and 30 named
feature columns. Identical dataset to Experiment 5.

NOTE on label encoding: this script encodes diagnosis 'M' (malignant) as
target = 0 and 'B' (benign) as target = 1.
```

## Reusable utilities (`ml_lab_utils`, from Experiment 1)

Inlined here so this notebook runs on its own without a separate `ml_lab_utils.py`.

In [ ]:
"""
ml_lab_utils.py
================
ICS1512 - Machine Learning Algorithms Laboratory
Reusable utility module used across ALL experiments.

Implements (per lab manual, Section 4):
    1. One reusable EDA function            -> generate_eda_summary()
    2. One reusable Regression function      -> train_evaluate_regression()
    3. One reusable Classification function  -> train_evaluate_classification()
    4. One reusable Regression metrics fn    -> regression_performance_metrics()
    5. One reusable Classification metrics   -> classification_performance_metrics()

Formatting rules enforced everywhere (per lab manual, Section 1):
    - Times New Roman, 15 pt for all text / legends
    - Bold, Times New Roman, 15 pt axis labels
    - Figures exported as .eps at 600 DPI (Section 3)

NOTE on fonts: "Times New Roman" itself is a proprietary Microsoft font and is
not installable on Linux. Liberation Serif is metrically-compatible (identical
glyph widths/kerning) and is registered here under the family name
"Times New Roman" so that rcParams['font.family'] = 'Times New Roman' works
transparently. On Windows/macOS, if the real Times New Roman is installed,
matplotlib will simply use that instead.
"""

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

# --------------------------------------------------------------------------
# 1. GLOBAL PLOT STYLE  (Section 1 of the manual)
# --------------------------------------------------------------------------
def set_plot_style(font_size=15):
    """
    Applies the mandatory lab formatting to every matplotlib figure:
        - Times New Roman (or metric-compatible Liberation Serif) font
        - 15 pt base font size
        - 15 pt Times New Roman legends
        - Bold, 15 pt, Times New Roman axis labels
    Call this once at the start of a notebook / script.
    """
    # Register Liberation Serif under the alias "Times New Roman" if the
    # genuine font is not present on this machine.
    installed_fonts = {f.name for f in fm.fontManager.ttflist}
    if "Times New Roman" not in installed_fonts:
        liberation_paths = [
            "/usr/share/fonts/truetype/liberation/LiberationSerif-Regular.ttf",
            "/usr/share/fonts/truetype/liberation/LiberationSerif-Bold.ttf",
            "/usr/share/fonts/truetype/liberation/LiberationSerif-Italic.ttf",
            "/usr/share/fonts/truetype/liberation/LiberationSerif-BoldItalic.ttf",
        ]
        for p in liberation_paths:
            if os.path.exists(p):
                fm.fontManager.addfont(p)
                # Force the registered family name to "Times New Roman"
                # (FontEntry is a frozen dataclass in modern matplotlib, so we
                # replace the last-added entry rather than mutate it in place)
                last = fm.fontManager.ttflist[-1]
                fm.fontManager.ttflist[-1] = fm.FontEntry(
                    fname=last.fname, name="Times New Roman",
                    style=last.style, variant=last.variant,
                    weight=last.weight, stretch=last.stretch, size=last.size,
                )

    plt.rcParams.update({
        "font.family": "Times New Roman",
        "font.size": font_size,
        "legend.fontsize": font_size,
        "legend.title_fontsize": font_size,
        "axes.labelsize": font_size,
        "axes.labelweight": "bold",
        "axes.titlesize": font_size,
        "axes.titleweight": "bold",
        "xtick.labelsize": font_size - 2,
        "ytick.labelsize": font_size - 2,
        "figure.titlesize": font_size + 2,
        "savefig.dpi": 600,
        "figure.dpi": 150,   # screen preview; export always forced to 600 (see save)
        "svg.fonttype": "none",
    })


def _bold_axis_labels(ax, xlabel=None, ylabel=None, title=None, fs=15):
    """Helper: apply Times New Roman / Bold / 15pt to a single axis explicitly."""
    fp_bold = fm.FontProperties(family="Times New Roman", weight="bold", size=fs)
    fp_reg = fm.FontProperties(family="Times New Roman", size=fs - 2)
    if xlabel is not None:
        ax.set_xlabel(xlabel, fontproperties=fp_bold)
    if ylabel is not None:
        ax.set_ylabel(ylabel, fontproperties=fp_bold)
    if title is not None:
        ax.set_title(title, fontproperties=fp_bold, fontsize=fs)
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontproperties(fp_reg)
    leg = ax.get_legend()
    if leg is not None:
        for txt in leg.get_texts():
            txt.set_fontproperties(fp_reg)


def _save_eps(fig, save_path, also_png=True):
    """Export a figure as .eps at 600 DPI (Section 3 of the manual).

    If also_png is True, an additional .png copy is saved alongside the .eps
    (same basename) purely so the figure can be embedded when compiling the
    LaTeX report with pdflatex/xelatex, which cannot rasterize .eps directly
    without Ghostscript. The .eps remains the official, mandated deliverable.
    """
    if save_path is None:
        return None
    if not save_path.lower().endswith(".eps"):
        save_path = os.path.splitext(save_path)[0] + ".eps"
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
    fig.savefig(save_path, format="eps", dpi=600, bbox_inches="tight")
    if also_png:
        png_path = os.path.splitext(save_path)[0] + ".png"
        fig.savefig(png_path, format="png", dpi=200, bbox_inches="tight")
    return save_path


# --------------------------------------------------------------------------
# 2. GENERIC EDA FUNCTION  (Section 4.1)  -> ONE consolidated 12-subplot figure
# --------------------------------------------------------------------------
def generate_eda_summary(df, target_col=None, dataset_name="Dataset",
                          save_path=None, figsize=(22, 16)):
    """
    Generic, reusable EDA function that works on ANY tabular dataset
    (classification, regression, or unlabeled). Produces ONE consolidated
    figure containing 12 EDA subplots on a single page, per Section 2 of the
    lab manual.

    Parameters
    ----------
    df : pandas.DataFrame
        The full dataset (features + target, if any).
    target_col : str or None
        Name of the target/label column, if present. If None, the function
        treats the dataset as unlabeled and adapts the 12-panel layout
        accordingly (no class-distribution / target-correlation panels).
    dataset_name : str
        Used in the figure's suptitle.
    save_path : str or None
        If given, the figure is exported as .eps @ 600 DPI to this path.
    figsize : tuple
        Overall figure size in inches.

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    set_plot_style()
    df = df.copy()

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
    if target_col in numeric_cols:
        numeric_cols.remove(target_col)
    if target_col in categorical_cols:
        categorical_cols.remove(target_col)

    is_classification_target = (
        target_col is not None and
        (df[target_col].dtype == "object" or df[target_col].nunique() <= 20)
    )

    # Pick the most "informative" numeric feature (highest variance) as the
    # representative single feature for panels 6/8/9/10, instead of blindly
    # using the first column (which can be degenerate/constant, e.g. corner
    # pixels in an image dataset such as MNIST/Digits).
    if numeric_cols:
        # Prefer genuinely continuous columns (more than 5 distinct values) so
        # binary/near-constant encoded columns (e.g. a 0/1 "sex" flag, or
        # constant corner pixels in image data) are not picked as the
        # representative single feature for panels 6/8/9/10.
        continuous_cols = [c for c in numeric_cols if df[c].nunique() > 5]
        candidate_cols = continuous_cols if continuous_cols else numeric_cols
        variances = df[candidate_cols].var().sort_values(ascending=False)
        top_var_cols = variances.index.tolist()
        feat_a = top_var_cols[0]
        feat_b = top_var_cols[1] if len(top_var_cols) > 1 else top_var_cols[0]
        kde_cols = top_var_cols[:4]
    else:
        feat_a = feat_b = None
        kde_cols = []

    fig = plt.figure(figsize=figsize)
    fig.suptitle(f"Exploratory Data Analysis Summary \u2013 {dataset_name}",
                 fontweight="bold", fontsize=17,
                 fontproperties=fm.FontProperties(family="Times New Roman",
                                                   weight="bold", size=17))
    gs = fig.add_gridspec(3, 4, hspace=0.55, wspace=0.4)
    axes = [fig.add_subplot(gs[i // 4, i % 4]) for i in range(12)]
    panel = 0

    # ---- Panel 1: Dataset overview (head / shape as a text table) ----
    ax = axes[panel]; panel += 1
    ax.axis("off")
    overview_txt = (
        f"Shape: {df.shape[0]} rows x {df.shape[1]} cols\n"
        f"Numeric features: {len(numeric_cols)}\n"
        f"Categorical features: {len(categorical_cols)}\n"
        f"Missing cells: {int(df.isnull().sum().sum())}\n"
        f"Duplicate rows: {int(df.duplicated().sum())}"
    )
    ax.text(0.02, 0.9, overview_txt, va="top", ha="left",
            fontproperties=fm.FontProperties(family="Times New Roman", size=13),
            transform=ax.transAxes)
    _bold_axis_labels(ax, title="1. Dataset Overview")

    # ---- Panel 2: Statistical summary heat-table (mean/std/min/max) ----
    ax = axes[panel]; panel += 1
    if len(numeric_cols) > 0:
        desc = df[numeric_cols].describe().T[["mean", "std", "min", "max"]]
        desc_norm = (desc - desc.min()) / (desc.max() - desc.min() + 1e-9)
        sns.heatmap(desc_norm.iloc[:8], annot=desc.iloc[:8].round(1), fmt="",
                    cmap="Blues", cbar=False, ax=ax,
                    annot_kws={"fontsize": 8, "fontfamily": "Times New Roman"})
    _bold_axis_labels(ax, title="2. Statistical Summary")

    # ---- Panel 3: Missing value analysis ----
    ax = axes[panel]; panel += 1
    miss = df.isnull().mean().sort_values(ascending=False) * 100
    if miss.sum() == 0:
        ax.text(0.5, 0.5, "No Missing Values", ha="center", va="center",
                fontproperties=fm.FontProperties(family="Times New Roman", size=14))
        ax.axis("off")
    else:
        miss[miss > 0].head(10).plot(kind="bar", ax=ax, color="#c0392b")
    _bold_axis_labels(ax, "Feature", "% Missing", "3. Missing Value Analysis")

    # ---- Panel 4: Class distribution / target distribution ----
    ax = axes[panel]; panel += 1
    if target_col is not None:
        if is_classification_target:
            df[target_col].value_counts().plot(kind="bar", ax=ax, color="#2980b9")
            _bold_axis_labels(ax, "Class", "Count", "4. Class Distribution")
        else:
            sns.histplot(df[target_col], kde=True, ax=ax, color="#2980b9")
            _bold_axis_labels(ax, target_col, "Frequency", "4. Target Distribution")
    else:
        ax.axis("off")
        ax.text(0.5, 0.5, "No target column supplied", ha="center", va="center",
                fontproperties=fm.FontProperties(family="Times New Roman", size=12))
        _bold_axis_labels(ax, title="4. Target Distribution")

    # ---- Panel 5: Correlation matrix (heatmap) ----
    ax = axes[panel]; panel += 1
    corr_cols = numeric_cols[:10] if len(numeric_cols) > 10 else numeric_cols
    if len(corr_cols) >= 2:
        sns.heatmap(df[corr_cols].corr(), cmap="coolwarm", center=0, ax=ax,
                    cbar=False, annot=len(corr_cols) <= 6, fmt=".2f",
                    annot_kws={"fontsize": 7})
    _bold_axis_labels(ax, title="5. Correlation Matrix")

    # ---- Panel 6: Feature distribution (histogram of 1st numeric feature) ----
    ax = axes[panel]; panel += 1
    if feat_a is not None:
        sns.histplot(df[feat_a], kde=True, ax=ax, color="#27ae60")
    _bold_axis_labels(ax, feat_a if feat_a else "", "Frequency",
                       "6. Feature Distribution")

    # ---- Panel 7: Box plot (outlier detection) across numeric features ----
    ax = axes[panel]; panel += 1
    if len(numeric_cols) > 0:
        plot_cols = numeric_cols[:6]
        df_scaled = (df[plot_cols] - df[plot_cols].mean()) / (df[plot_cols].std() + 1e-9)
        sns.boxplot(data=df_scaled, ax=ax, color="#f39c12")
        ax.tick_params(axis="x", rotation=45)
    _bold_axis_labels(ax, "Feature", "Standardized Value", "7. Box Plot (Outliers)")

    # ---- Panel 8: Violin plot ----
    ax = axes[panel]; panel += 1
    if feat_a is not None:
        sns.violinplot(y=df[feat_a], ax=ax, color="#8e44ad")
    _bold_axis_labels(ax, "", feat_a if feat_a else "",
                       "8. Violin Plot")

    # ---- Panel 9: Scatter plot (feature 1 vs feature 2, hued by target) ----
    ax = axes[panel]; panel += 1
    if feat_a is not None and feat_b is not None:
        hue = df[target_col] if (target_col and is_classification_target) else None
        sns.scatterplot(x=df[feat_a], y=df[feat_b],
                         hue=hue, ax=ax, palette="Set2", legend=False, s=18)
    _bold_axis_labels(ax, feat_a if feat_a else "", feat_b if feat_b else "",
                       "9. Scatter Plot")

    # ---- Panel 10: Q-Q plot (normality check on highest-variance feature) ----
    ax = axes[panel]; panel += 1
    if feat_a is not None:
        stats.probplot(df[feat_a].dropna(), dist="norm", plot=ax)
        ax.get_lines()[0].set_markerfacecolor("#2980b9")
        ax.get_lines()[0].set_markeredgecolor("#2980b9")
        ax.get_lines()[1].set_color("#c0392b")
    _bold_axis_labels(ax, "Theoretical Quantiles", "Sample Quantiles", "10. Q-Q Plot")

    # ---- Panel 11: KDE / density plot overlay of top numeric features ----
    ax = axes[panel]; panel += 1
    for c in kde_cols:
        sns.kdeplot(df[c], ax=ax, label=c, linewidth=1.5)
    if kde_cols:
        ax.legend(prop=fm.FontProperties(family="Times New Roman", size=9))
    _bold_axis_labels(ax, "Value", "Density", "11. KDE / Density Plot")

    # ---- Panel 12: Feature importance / variance plot ----
    ax = axes[panel]; panel += 1
    if len(numeric_cols) > 0:
        var = df[numeric_cols].var().sort_values(ascending=False).head(8)
        var.plot(kind="barh", ax=ax, color="#16a085")
        ax.invert_yaxis()
    _bold_axis_labels(ax, "Variance", "Feature", "12. Variance / Importance Plot")

    for ax in axes:
        _bold_axis_labels(ax)  # re-apply tick font in case a plotting call reset it

    saved = _save_eps(fig, save_path)
    if saved:
        print(f"[generate_eda_summary] Figure saved -> {saved} (600 DPI, EPS)")
    return fig


# --------------------------------------------------------------------------
# 3. GENERIC REGRESSION TRAIN/EVAL FUNCTION  (Section 4.2)
# --------------------------------------------------------------------------
def train_evaluate_regression(models: dict, X_train, X_test, y_train, y_test,
                               scale=False, verbose=True):
    """
    Reusable function to train and evaluate an arbitrary set of regression
    models on the same train/test split.

    Parameters
    ----------
    models : dict {name: sklearn-estimator}
    X_train, X_test, y_train, y_test : array-like
    scale : bool -> StandardScaler applied when True (fit on train only)
    verbose : bool -> print per-model metrics as they are computed

    Returns
    -------
    results_df : pandas.DataFrame  (one row per model, sorted by R2 desc)
    fitted_models : dict {name: fitted estimator}
    """
    from sklearn.preprocessing import StandardScaler

    if scale:
        scaler = StandardScaler().fit(X_train)
        X_train = scaler.transform(X_train)
        X_test = scaler.transform(X_test)

    rows, fitted_models = [], {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        metrics = regression_performance_metrics(y_test, y_pred, model_name=name,
                                                   verbose=verbose, return_dict=True)
        rows.append(metrics)
        fitted_models[name] = model

    results_df = pd.DataFrame(rows).set_index("Model").sort_values("R2", ascending=False)
    return results_df, fitted_models


# --------------------------------------------------------------------------
# 4. GENERIC CLASSIFICATION TRAIN/EVAL FUNCTION  (Section 4.3)
# --------------------------------------------------------------------------
def train_evaluate_classification(models: dict, X_train, X_test, y_train, y_test,
                                   scale=False, average="weighted", verbose=True):
    """
    Reusable function to train and evaluate an arbitrary set of classification
    models on the same train/test split.

    Returns
    -------
    results_df : pandas.DataFrame  (one row per model, sorted by Accuracy desc)
    fitted_models : dict {name: fitted estimator}
    """
    from sklearn.preprocessing import StandardScaler

    if scale:
        scaler = StandardScaler().fit(X_train)
        X_train = scaler.transform(X_train)
        X_test = scaler.transform(X_test)

    rows, fitted_models = [], {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = None
        if hasattr(model, "predict_proba"):
            try:
                y_proba = model.predict_proba(X_test)
            except Exception:
                y_proba = None
        metrics = classification_performance_metrics(
            y_test, y_pred, y_proba=y_proba, model_name=name,
            average=average, verbose=verbose, return_dict=True, plot=False
        )
        rows.append(metrics)
        fitted_models[name] = model

    results_df = pd.DataFrame(rows).set_index("Model").sort_values("Accuracy", ascending=False)
    return results_df, fitted_models


# --------------------------------------------------------------------------
# 5. GENERIC REGRESSION METRICS FUNCTION  (Section 4.4)
# --------------------------------------------------------------------------
def regression_performance_metrics(y_true, y_pred, model_name="Model",
                                    verbose=True, return_dict=False):
    """
    Computes and displays ALL standard regression performance metrics:
    MAE, MSE, RMSE, R2, Adjusted R2 (n only), MAPE.
    """
    from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                                  r2_score, mean_absolute_percentage_error)

    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100

    if verbose:
        print(f"--- Regression Metrics: {model_name} ---")
        print(f"  MAE  : {mae:.4f}")
        print(f"  MSE  : {mse:.4f}")
        print(f"  RMSE : {rmse:.4f}")
        print(f"  R2   : {r2:.4f}")
        print(f"  MAPE : {mape:.2f}%\n")

    result = {"Model": model_name, "MAE": mae, "MSE": mse,
              "RMSE": rmse, "R2": r2, "MAPE(%)": mape}
    return result if return_dict else pd.DataFrame([result]).set_index("Model")


# --------------------------------------------------------------------------
# 6. GENERIC CLASSIFICATION METRICS FUNCTION  (Section 4.5)
# --------------------------------------------------------------------------
def classification_performance_metrics(y_true, y_pred, y_proba=None,
                                        model_name="Model", average="weighted",
                                        verbose=True, return_dict=False,
                                        plot=True, save_path=None):
    """
    Computes and displays ALL standard classification performance metrics:
    Accuracy, Precision, Recall, F1-score, ROC-AUC (binary/multiclass ovr),
    and (optionally) plots the confusion matrix using the mandatory lab
    formatting (Times New Roman, bold 15pt axis labels).
    """
    from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                                  f1_score, roc_auc_score, confusion_matrix)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average=average, zero_division=0)
    rec = recall_score(y_true, y_pred, average=average, zero_division=0)
    f1 = f1_score(y_true, y_pred, average=average, zero_division=0)

    roc_auc = np.nan
    if y_proba is not None:
        try:
            n_classes = y_proba.shape[1]
            if n_classes == 2:
                roc_auc = roc_auc_score(y_true, y_proba[:, 1])
            else:
                roc_auc = roc_auc_score(y_true, y_proba, multi_class="ovr",
                                         average=average)
        except Exception:
            roc_auc = np.nan

    if verbose:
        print(f"--- Classification Metrics: {model_name} ---")
        print(f"  Accuracy  : {acc:.4f}")
        print(f"  Precision : {prec:.4f}")
        print(f"  Recall    : {rec:.4f}")
        print(f"  F1-score  : {f1:.4f}")
        print(f"  ROC-AUC   : {roc_auc:.4f}" if not np.isnan(roc_auc) else "  ROC-AUC   : N/A")
        print()

    if plot:
        set_plot_style()
        cm = confusion_matrix(y_true, y_pred)
        fig, ax = plt.subplots(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                    annot_kws={"fontfamily": "Times New Roman", "fontsize": 13})
        _bold_axis_labels(ax, "Predicted Label", "True Label",
                           f"Confusion Matrix \u2013 {model_name}")
        _save_eps(fig, save_path)

    result = {"Model": model_name, "Accuracy": acc, "Precision": prec,
              "Recall": rec, "F1-score": f1, "ROC-AUC": roc_auc}
    return result if return_dict else pd.DataFrame([result]).set_index("Model")


In [1]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                      cross_validate, GridSearchCV)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (BaggingClassifier, AdaBoostClassifier,
                               GradientBoostingClassifier, StackingClassifier)
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, roc_curve, confusion_matrix)


warnings.filterwarnings("ignore")
RANDOM_STATE = 42

FIG_DIR = "figures"
RES_DIR = "results"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

set_plot_style()


## 1. LOAD DATASET AND ENCODE CLASS LABELS

In [2]:
DATA_PATH = "breast_cancer_data.csv"  # <-- upload this file to the notebook's working directory
raw = pd.read_csv(DATA_PATH)

drop_cols = [c for c in raw.columns if c.lower() == "id" or "unnamed" in c.lower()]
df = raw.drop(columns=drop_cols).copy()

feature_names = [c for c in df.columns if c != "diagnosis"]

# target: 0 = malignant (M), 1 = benign (B)
df["target"] = df["diagnosis"].map({"M": 0, "B": 1})
df["diagnosis"] = df["diagnosis"].map({"M": "Malignant", "B": "Benign"})

print("Dataset shape:", df.shape)
print(df["diagnosis"].value_counts())
print("Missing values:", int(df[feature_names].isnull().sum().sum()))

Dataset shape: (569, 32)
diagnosis
Benign       357
Malignant    212
Name: count, dtype: int64
Missing values: 0


## 2. EDA (reusable function from Experiment 1)

In [3]:
eda_df = df.drop(columns=["target"])
generate_eda_summary(
    eda_df, target_col="diagnosis", dataset_name="Wisconsin Breast Cancer (Exp. 6)",
    save_path=f"{FIG_DIR}/eda_breast_cancer.eps"
)
plt.close("all")

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


[generate_eda_summary] Figure saved -> figures/eda_breast_cancer.eps (600 DPI, EPS)


## 3. TRAIN / TEST SPLIT (80-20, stratified)

In [4]:
X = df[feature_names].values
y = df["target"].values  # 0 = malignant, 1 = benign

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


def time_fit_predict(model, Xtr, Xte, ytr):
    t0 = time.perf_counter()
    model.fit(Xtr, ytr)
    train_t = time.perf_counter() - t0
    t0 = time.perf_counter()
    y_pred = model.predict(Xte)
    pred_t = time.perf_counter() - t0
    return y_pred, train_t, pred_t

## 4. BAGGING CLASSIFIER (base estimator: Decision Tree)

In [5]:
baseline_models = {
    "Bagging (Baseline)": BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
        random_state=RANDOM_STATE)
}
baseline_results_df, baseline_fitted = train_evaluate_classification(
    baseline_models, X_train, X_test, y_train, y_test, scale=False
)
print("\n=== Baseline Bagging (default hyperparameters) ===")
print(baseline_results_df)

bagging_param_grid = {
    "n_estimators": [10, 50, 100],
    "max_samples": [0.5, 0.7, 1.0],
    "max_features": [0.5, 0.7, 1.0],
}

print("\n[Bagging] Starting GridSearchCV (5-fold)...")
t0 = time.perf_counter()
bagging_grid = GridSearchCV(
    BaggingClassifier(estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
                       random_state=RANDOM_STATE),
    bagging_param_grid, cv=cv_strategy, scoring="accuracy", n_jobs=1)
bagging_grid.fit(X_train, y_train)
bagging_grid_time = time.perf_counter() - t0
print(f"[Bagging] GridSearchCV done in {bagging_grid_time:.1f}s")
print("Best params:", bagging_grid.best_params_)
print("Best CV accuracy:", bagging_grid.best_score_)

# Table 1: n_estimators x max_samples summary (best over max_features)
bagging_cv_results = pd.DataFrame(bagging_grid.cv_results_)
bagging_table1_rows = []
for n_est in [10, 50, 100]:
    for max_samp in [0.5, 0.7, 1.0]:
        mask = ((bagging_cv_results["param_n_estimators"] == n_est) &
                 (bagging_cv_results["param_max_samples"] == max_samp))
        subset = bagging_cv_results[mask]
        if subset.empty:
            continue
        best_row = subset.loc[subset["mean_test_score"].idxmax()]
        params = best_row["params"]
        model = BaggingClassifier(estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
                                   random_state=RANDOM_STATE, **params)
        cv_res = cross_validate(model, X_train, y_train, cv=cv_strategy,
                                 scoring=["accuracy", "f1"])
        bagging_table1_rows.append({
            "n_estimators": n_est,
            "max_samples": max_samp,
            "Avg CV Accuracy (%)": cv_res["test_accuracy"].mean() * 100,
            "Avg CV F1 Score": cv_res["test_f1"].mean(),
        })
bagging_table1_df = pd.DataFrame(bagging_table1_rows)
bagging_table1_df.to_csv(f"{RES_DIR}/bagging_hyperparameter_evaluation.csv", index=False)
print("\n=== Table 1: Bagging Hyperparameter Evaluation (5-Fold CV) ===")
print(bagging_table1_df)

best_bagging = bagging_grid.best_estimator_
y_pred_bag, bag_train_t, bag_pred_t = time_fit_predict(best_bagging, X_train, X_test, y_train)
y_proba_bag = best_bagging.predict_proba(X_test)
bagging_metrics = classification_performance_metrics(
    y_test, y_pred_bag, y_proba_bag, model_name="Bagging (Tuned)",
    return_dict=True, plot=True, save_path=f"{FIG_DIR}/cm_bagging.eps"
)
bagging_metrics["Training Time (s)"] = bag_train_t
print("\n=== Tuned Bagging Performance ===")
print(bagging_metrics)

--- Classification Metrics: Bagging (Baseline) ---
  Accuracy  : 0.9298
  Precision : 0.9298
  Recall    : 0.9298
  F1-score  : 0.9298
  ROC-AUC   : 0.9899


=== Baseline Bagging (default hyperparameters) ===
                    Accuracy  Precision    Recall  F1-score   ROC-AUC
Model                                                                
Bagging (Baseline)  0.929825   0.929825  0.929825  0.929825  0.989914

[Bagging] Starting GridSearchCV (5-fold)...


[Bagging] GridSearchCV done in 17.8s
Best params: {'max_features': 0.5, 'max_samples': 0.5, 'n_estimators': 50}
Best CV accuracy: 0.9604395604395606



=== Table 1: Bagging Hyperparameter Evaluation (5-Fold CV) ===
   n_estimators  max_samples  Avg CV Accuracy (%)  Avg CV F1 Score
0            10          0.5            95.164835         0.961019
1            10          0.7            95.164835         0.961139
2            10          1.0            95.604396         0.964839
3            50          0.5            96.043956         0.968138
4            50          0.7            95.824176         0.966520
5            50          1.0            96.043956         0.968325
6           100          0.5            95.384615         0.962975
7           100          0.7            96.043956         0.968288
8           100          1.0            96.043956         0.968418
--- Classification Metrics: Bagging (Tuned) ---
  Accuracy  : 0.9561
  Precision : 0.9561
  Recall    : 0.9561
  F1-score  : 0.9560
  ROC-AUC   : 0.9931




=== Tuned Bagging Performance ===
{'Model': 'Bagging (Tuned)', 'Accuracy': 0.956140350877193, 'Precision': 0.9560729421281235, 'Recall': 0.956140350877193, 'F1-score': 0.9560273762928302, 'ROC-AUC': 0.9930555555555556, 'Training Time (s)': 0.0972587599999315}


## 5. BOOSTING CLASSIFIERS (AdaBoost and Gradient Boosting)

In [6]:
boosting_baseline = {
    "AdaBoost (Baseline)": AdaBoostClassifier(random_state=RANDOM_STATE),
    "Gradient Boosting (Baseline)": GradientBoostingClassifier(random_state=RANDOM_STATE),
}
boosting_baseline_df, boosting_baseline_fitted = train_evaluate_classification(
    boosting_baseline, X_train, X_test, y_train, y_test, scale=False
)
print("\n=== Baseline Boosting Models (default hyperparameters) ===")
print(boosting_baseline_df)

# AdaBoost hyperparameter tuning
ada_param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1.0],
}
print("\n[AdaBoost] Starting GridSearchCV (5-fold)...")
t0 = time.perf_counter()
ada_grid = GridSearchCV(AdaBoostClassifier(random_state=RANDOM_STATE), ada_param_grid,
                         cv=cv_strategy, scoring="accuracy", n_jobs=1)
ada_grid.fit(X_train, y_train)
ada_grid_time = time.perf_counter() - t0
print(f"[AdaBoost] GridSearchCV done in {ada_grid_time:.1f}s")
print("Best params:", ada_grid.best_params_)
print("Best CV accuracy:", ada_grid.best_score_)

# Gradient Boosting hyperparameter tuning
gb_param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1.0],
    "max_depth": [2, 3, 5],
}
print("\n[Gradient Boosting] Starting GridSearchCV (5-fold)...")
t0 = time.perf_counter()
gb_grid = GridSearchCV(GradientBoostingClassifier(random_state=RANDOM_STATE), gb_param_grid,
                        cv=cv_strategy, scoring="accuracy", n_jobs=1)
gb_grid.fit(X_train, y_train)
gb_grid_time = time.perf_counter() - t0
print(f"[Gradient Boosting] GridSearchCV done in {gb_grid_time:.1f}s")
print("Best params:", gb_grid.best_params_)
print("Best CV accuracy:", gb_grid.best_score_)

# Table 2: n_estimators x learning_rate summary, using AdaBoost as the
# representative boosting algorithm for this table shape (max_depth held at
# GB's own grid separately, reported alongside)
ada_cv_results = pd.DataFrame(ada_grid.cv_results_)
boosting_table2_rows = []
for n_est in [50, 100, 200]:
    for lr in [0.01, 0.1, 1.0]:
        mask = ((ada_cv_results["param_n_estimators"] == n_est) &
                 (ada_cv_results["param_learning_rate"] == lr))
        subset = ada_cv_results[mask]
        if subset.empty:
            continue
        params = subset.iloc[0]["params"]
        model = AdaBoostClassifier(random_state=RANDOM_STATE, **params)
        cv_res = cross_validate(model, X_train, y_train, cv=cv_strategy,
                                 scoring=["accuracy", "f1"])
        boosting_table2_rows.append({
            "Algorithm": "AdaBoost",
            "n_estimators": n_est,
            "learning_rate": lr,
            "Avg CV Accuracy (%)": cv_res["test_accuracy"].mean() * 100,
            "Avg CV F1 Score": cv_res["test_f1"].mean(),
        })
boosting_table2_df = pd.DataFrame(boosting_table2_rows)
boosting_table2_df.to_csv(f"{RES_DIR}/boosting_hyperparameter_evaluation.csv", index=False)
print("\n=== Table 2: Boosting (AdaBoost) Hyperparameter Evaluation (5-Fold CV) ===")
print(boosting_table2_df)

# Choose the better of tuned AdaBoost vs tuned Gradient Boosting as "Boosting"
best_ada = ada_grid.best_estimator_
best_gb = gb_grid.best_estimator_
if ada_grid.best_score_ >= gb_grid.best_score_:
    best_boosting = best_ada
    best_boosting_name = "AdaBoost (Tuned)"
else:
    best_boosting = best_gb
    best_boosting_name = "Gradient Boosting (Tuned)"
print(f"\nSelected as overall 'Boosting' model: {best_boosting_name}")

y_pred_boost, boost_train_t, boost_pred_t = time_fit_predict(best_boosting, X_train, X_test, y_train)
y_proba_boost = best_boosting.predict_proba(X_test)
boosting_metrics = classification_performance_metrics(
    y_test, y_pred_boost, y_proba_boost, model_name="Boosting (Tuned)",
    return_dict=True, plot=True, save_path=f"{FIG_DIR}/cm_boosting.eps"
)
boosting_metrics["Training Time (s)"] = boost_train_t
boosting_metrics["Selected Algorithm"] = best_boosting_name
print("\n=== Tuned Boosting Performance ===")
print(boosting_metrics)

--- Classification Metrics: AdaBoost (Baseline) ---
  Accuracy  : 0.9561
  Precision : 0.9569
  Recall    : 0.9561
  F1-score  : 0.9558
  ROC-AUC   : 0.9825



--- Classification Metrics: Gradient Boosting (Baseline) ---
  Accuracy  : 0.9561
  Precision : 0.9569
  Recall    : 0.9561
  F1-score  : 0.9558
  ROC-AUC   : 0.9907


=== Baseline Boosting Models (default hyperparameters) ===
                              Accuracy  Precision   Recall  F1-score   ROC-AUC
Model                                                                         
AdaBoost (Baseline)            0.95614   0.956869  0.95614  0.955776  0.982474
Gradient Boosting (Baseline)   0.95614   0.956869  0.95614  0.955776  0.990741

[AdaBoost] Starting GridSearchCV (5-fold)...


[AdaBoost] GridSearchCV done in 11.1s
Best params: {'learning_rate': 1.0, 'n_estimators': 100}
Best CV accuracy: 0.9802197802197803

[Gradient Boosting] Starting GridSearchCV (5-fold)...


[Gradient Boosting] GridSearchCV done in 36.6s
Best params: {'learning_rate': 0.1, 'max_depth': 2, 'n_estimators': 200}
Best CV accuracy: 0.9670329670329672



=== Table 2: Boosting (AdaBoost) Hyperparameter Evaluation (5-Fold CV) ===
  Algorithm  n_estimators  learning_rate  Avg CV Accuracy (%)  Avg CV F1 Score
0  AdaBoost            50           0.01            92.527473         0.940745
1  AdaBoost            50           0.10            95.384615         0.963289
2  AdaBoost            50           1.00            96.703297         0.974002
3  AdaBoost           100           0.01            93.406593         0.947975
4  AdaBoost           100           0.10            96.043956         0.968602
5  AdaBoost           100           1.00            98.021978         0.984378
6  AdaBoost           200           0.01            93.626374         0.949600
7  AdaBoost           200           0.10            97.142857         0.977269
8  AdaBoost           200           1.00            97.802198         0.982608

Selected as overall 'Boosting' model: AdaBoost (Tuned)


--- Classification Metrics: Boosting (Tuned) ---
  Accuracy  : 0.9561
  Precision : 0.9569
  Recall    : 0.9561
  F1-score  : 0.9558
  ROC-AUC   : 0.9818


=== Tuned Boosting Performance ===
{'Model': 'Boosting (Tuned)', 'Accuracy': 0.956140350877193, 'Precision': 0.9568690958164642, 'Recall': 0.956140350877193, 'F1-score': 0.9557756825927252, 'ROC-AUC': 0.9818121693121693, 'Training Time (s)': 0.249369932000036, 'Selected Algorithm': 'AdaBoost (Tuned)'}


## 6. STACKED ENSEMBLE (Base: SVM, Naive Bayes, Decision Tree; Meta: LogReg)

In [7]:
base_learners = [
    ("svm", SVC(probability=True, random_state=RANDOM_STATE)),
    ("nb", GaussianNB()),
    ("dt", DecisionTreeClassifier(random_state=RANDOM_STATE)),
]
stacking_baseline = StackingClassifier(
    estimators=base_learners,
    final_estimator=LogisticRegression(max_iter=2000),
    cv=cv_strategy
)
stacking_models = {"Stacked Ensemble (Baseline)": stacking_baseline}
stacking_baseline_df, stacking_baseline_fitted = train_evaluate_classification(
    stacking_models, X_train, X_test, y_train, y_test, scale=False
)
print("\n=== Baseline Stacked Ensemble ===")
print(stacking_baseline_df)

# Compare a couple of base-model / meta-learner combinations (Table 3)
stacking_configs = {
    "SVM+NB+DT / LogReg": StackingClassifier(
        estimators=[("svm", SVC(probability=True, random_state=RANDOM_STATE)),
                    ("nb", GaussianNB()),
                    ("dt", DecisionTreeClassifier(random_state=RANDOM_STATE))],
        final_estimator=LogisticRegression(max_iter=2000), cv=cv_strategy),
    "SVM+NB+DT / DecisionTree": StackingClassifier(
        estimators=[("svm", SVC(probability=True, random_state=RANDOM_STATE)),
                    ("nb", GaussianNB()),
                    ("dt", DecisionTreeClassifier(random_state=RANDOM_STATE))],
        final_estimator=DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE),
        cv=cv_strategy),
    "NB+DT / LogReg": StackingClassifier(
        estimators=[("nb", GaussianNB()),
                    ("dt", DecisionTreeClassifier(random_state=RANDOM_STATE))],
        final_estimator=LogisticRegression(max_iter=2000), cv=cv_strategy),
}

print("\n[Stacking] Evaluating base-model / meta-learner combinations (5-fold CV)...")
stacking_table3_rows = []
for combo_name, model in stacking_configs.items():
    t0 = time.perf_counter()
    cv_res = cross_validate(model, X_train, y_train, cv=cv_strategy, scoring=["accuracy", "f1"])
    combo_time = time.perf_counter() - t0
    base_str, meta_str = combo_name.split(" / ")
    stacking_table3_rows.append({
        "Base Models": base_str,
        "Meta Learner": meta_str,
        "Avg CV Accuracy (%)": cv_res["test_accuracy"].mean() * 100,
        "Avg CV F1 Score": cv_res["test_f1"].mean(),
    })
    print(f"  {combo_name}: CV Accuracy = {cv_res['test_accuracy'].mean()*100:.2f}%, "
          f"time = {combo_time:.1f}s")

stacking_table3_df = pd.DataFrame(stacking_table3_rows)
stacking_table3_df.to_csv(f"{RES_DIR}/stacking_hyperparameter_evaluation.csv", index=False)
print("\n=== Table 3: Stacked Ensemble Evaluation (5-Fold CV) ===")
print(stacking_table3_df)

best_combo_idx = stacking_table3_df["Avg CV Accuracy (%)"].idxmax()
best_combo_name = list(stacking_configs.keys())[best_combo_idx]
best_stacking = stacking_configs[best_combo_name]
print(f"\nBest stacking configuration: {best_combo_name}")

y_pred_stack, stack_train_t, stack_pred_t = time_fit_predict(best_stacking, X_train, X_test, y_train)
y_proba_stack = best_stacking.predict_proba(X_test)
stacking_metrics = classification_performance_metrics(
    y_test, y_pred_stack, y_proba_stack, model_name="Stacked Ensemble (Tuned)",
    return_dict=True, plot=True, save_path=f"{FIG_DIR}/cm_stacking.eps"
)
stacking_metrics["Training Time (s)"] = stack_train_t
stacking_metrics["Best Configuration"] = best_combo_name
print("\n=== Tuned Stacked Ensemble Performance ===")
print(stacking_metrics)

--- Classification Metrics: Stacked Ensemble (Baseline) ---
  Accuracy  : 0.9561
  Precision : 0.9569
  Recall    : 0.9561
  F1-score  : 0.9558
  ROC-AUC   : 0.9858


=== Baseline Stacked Ensemble ===
                             Accuracy  Precision   Recall  F1-score  ROC-AUC
Model                                                                       
Stacked Ensemble (Baseline)   0.95614   0.956869  0.95614  0.955776  0.98578

[Stacking] Evaluating base-model / meta-learner combinations (5-fold CV)...


  SVM+NB+DT / LogReg: CV Accuracy = 95.38%, time = 0.4s


  SVM+NB+DT / DecisionTree: CV Accuracy = 94.07%, time = 0.4s
  NB+DT / LogReg: CV Accuracy = 94.29%, time = 0.2s

=== Table 3: Stacked Ensemble Evaluation (5-Fold CV) ===
  Base Models  Meta Learner  Avg CV Accuracy (%)  Avg CV F1 Score
0   SVM+NB+DT        LogReg            95.384615         0.963523
1   SVM+NB+DT  DecisionTree            94.065934         0.952771
2       NB+DT        LogReg            94.285714         0.955075

Best stacking configuration: SVM+NB+DT / LogReg


--- Classification Metrics: Stacked Ensemble (Tuned) ---
  Accuracy  : 0.9561
  Precision : 0.9569
  Recall    : 0.9561
  F1-score  : 0.9558
  ROC-AUC   : 0.9858


=== Tuned Stacked Ensemble Performance ===
{'Model': 'Stacked Ensemble (Tuned)', 'Accuracy': 0.956140350877193, 'Precision': 0.9568690958164642, 'Recall': 0.956140350877193, 'F1-score': 0.9557756825927252, 'ROC-AUC': 0.9857804232804233, 'Training Time (s)': 0.09182921199999328, 'Best Configuration': 'SVM+NB+DT / LogReg'}


## 7. HYPERPARAMETER TUNING RESULTS SUMMARY

In [8]:
tuning_summary = pd.DataFrame({
    "Bagging": {
        "Search Method": "GridSearchCV (5-fold)",
        "Best Parameters": str(bagging_grid.best_params_),
        "Best CV Accuracy": bagging_grid.best_score_,
    },
    "AdaBoost": {
        "Search Method": "GridSearchCV (5-fold)",
        "Best Parameters": str(ada_grid.best_params_),
        "Best CV Accuracy": ada_grid.best_score_,
    },
    "Gradient Boosting": {
        "Search Method": "GridSearchCV (5-fold)",
        "Best Parameters": str(gb_grid.best_params_),
        "Best CV Accuracy": gb_grid.best_score_,
    },
    "Stacked Ensemble": {
        "Search Method": "Manual combo search (5-fold CV)",
        "Best Parameters": best_combo_name,
        "Best CV Accuracy": stacking_table3_df["Avg CV Accuracy (%)"].max() / 100,
    },
}).T
tuning_summary.to_csv(f"{RES_DIR}/hyperparameter_tuning_results.csv")
print("\n=== Hyperparameter Tuning Results Summary ===")
print(tuning_summary)


=== Hyperparameter Tuning Results Summary ===
                                     Search Method  \
Bagging                      GridSearchCV (5-fold)   
AdaBoost                     GridSearchCV (5-fold)   
Gradient Boosting            GridSearchCV (5-fold)   
Stacked Ensemble   Manual combo search (5-fold CV)   

                                                     Best Parameters  \
Bagging            {'max_features': 0.5, 'max_samples': 0.5, 'n_e...   
AdaBoost                 {'learning_rate': 1.0, 'n_estimators': 100}   
Gradient Boosting  {'learning_rate': 0.1, 'max_depth': 2, 'n_esti...   
Stacked Ensemble                                  SVM+NB+DT / LogReg   

                  Best CV Accuracy  
Bagging                    0.96044  
AdaBoost                   0.98022  
Gradient Boosting         0.967033  
Stacked Ensemble          0.953846  


## 8. PERFORMANCE COMPARISON TABLE (Table 4)

In [9]:
performance_comparison = pd.DataFrame([bagging_metrics, boosting_metrics, stacking_metrics])
performance_comparison = performance_comparison[
    ["Model", "Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC", "Training Time (s)"]
].set_index("Model")
performance_comparison.to_csv(f"{RES_DIR}/performance_comparison.csv")
print("\n=== Table 4: Performance Comparison of Ensemble Models ===")
print(performance_comparison)

fig, ax = plt.subplots(figsize=(9, 5))
performance_comparison[["Accuracy", "Precision", "Recall", "F1-score"]].plot(kind="bar", ax=ax)
ax.legend(prop=fm.FontProperties(family="Times New Roman", size=11))
_bold_axis_labels(ax, "Model", "Score", "Ensemble Model Comparison")
plt.xticks(rotation=15, ha="right")
_save_eps(fig, f"{FIG_DIR}/ensemble_comparison_bar.eps")
plt.close(fig)

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.



=== Table 4: Performance Comparison of Ensemble Models ===
                          Accuracy  Precision   Recall  F1-score   ROC-AUC  \
Model                                                                        
Bagging (Tuned)            0.95614   0.956073  0.95614  0.956027  0.993056   
Boosting (Tuned)           0.95614   0.956869  0.95614  0.955776  0.981812   
Stacked Ensemble (Tuned)   0.95614   0.956869  0.95614  0.955776  0.985780   

                          Training Time (s)  
Model                                        
Bagging (Tuned)                    0.097259  
Boosting (Tuned)                   0.249370  
Stacked Ensemble (Tuned)           0.091829  


## 9. 5-FOLD CROSS-VALIDATION COMPARISON (all three tuned ensembles)

In [10]:
cv_bag = cross_validate(best_bagging, X_train, y_train, cv=cv_strategy, scoring="accuracy")
cv_boost = cross_validate(best_boosting, X_train, y_train, cv=cv_strategy, scoring="accuracy")
cv_stack = cross_validate(best_stacking, X_train, y_train, cv=cv_strategy, scoring="accuracy")

cv_table = pd.DataFrame({
    "Fold": [f"Fold {i+1}" for i in range(5)] + ["Average"],
    "Bagging": list(cv_bag["test_score"]) + [cv_bag["test_score"].mean()],
    "Boosting": list(cv_boost["test_score"]) + [cv_boost["test_score"].mean()],
    "Stacked Ensemble": list(cv_stack["test_score"]) + [cv_stack["test_score"].mean()],
}).set_index("Fold")
cv_table.to_csv(f"{RES_DIR}/cv_accuracy_comparison.csv")
print("\n=== 5-Fold Cross-Validation Comparison ===")
print(cv_table)

fig, ax = plt.subplots(figsize=(8, 5))
folds = np.arange(1, 6)
ax.plot(folds, cv_bag["test_score"], marker="o", label="Bagging", color="#2980b9")
ax.plot(folds, cv_boost["test_score"], marker="s", label="Boosting", color="#c0392b")
ax.plot(folds, cv_stack["test_score"], marker="^", label="Stacked Ensemble", color="#16a085")
ax.legend()
_bold_axis_labels(ax, "Fold", "Accuracy", "5-Fold Cross-Validation Accuracy Comparison")
_save_eps(fig, f"{FIG_DIR}/cv_accuracy_comparison.eps")
plt.close(fig)

# Fold-to-fold standard deviation as a stability proxy
stability_df = pd.DataFrame({
    "Model": ["Bagging", "Boosting", "Stacked Ensemble"],
    "CV Mean Accuracy": [cv_bag["test_score"].mean(), cv_boost["test_score"].mean(),
                          cv_stack["test_score"].mean()],
    "CV Std Dev (Stability)": [cv_bag["test_score"].std(), cv_boost["test_score"].std(),
                                cv_stack["test_score"].std()],
}).set_index("Model")
stability_df.to_csv(f"{RES_DIR}/stability_comparison.csv")
print("\n=== Stability Comparison (CV std dev) ===")
print(stability_df)

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.



=== 5-Fold Cross-Validation Comparison ===
          Bagging  Boosting  Stacked Ensemble
Fold                                         
Fold 1   0.956044  0.989011          0.934066
Fold 2   0.956044  1.000000          0.978022
Fold 3   0.934066  0.945055          0.934066
Fold 4   0.967033  0.978022          0.934066
Fold 5   0.989011  0.989011          0.989011
Average  0.960440  0.980220          0.953846



=== Stability Comparison (CV std dev) ===
                  CV Mean Accuracy  CV Std Dev (Stability)
Model                                                     
Bagging                   0.960440                0.017855
Boosting                  0.980220                0.018906
Stacked Ensemble          0.953846                0.024474


## 10. ROC CURVES (all three tuned ensembles)

In [11]:
fig, ax = plt.subplots(figsize=(7, 6))
for name, model in [("Bagging (Tuned)", best_bagging),
                     (f"Boosting (Tuned - {best_boosting_name.split(' ')[0]})", best_boosting),
                     ("Stacked Ensemble (Tuned)", best_stacking)]:
    proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})", linewidth=1.8)
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
ax.legend(fontsize=9)
_bold_axis_labels(ax, "False Positive Rate", "True Positive Rate", "ROC Curves")
_save_eps(fig, f"{FIG_DIR}/roc_curves.eps")
plt.close(fig)

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


## 11. BIAS-VARIANCE ANALYSIS: single tree vs bagging vs boosting (train/val gap)

In [12]:
bias_variance_rows = []
single_tree = DecisionTreeClassifier(random_state=RANDOM_STATE)
for name, model in [("Single Decision Tree", single_tree),
                     ("Bagging (Tuned)", best_bagging),
                     ("Boosting (Tuned)", best_boosting),
                     ("Stacked Ensemble (Tuned)", best_stacking)]:
    cv_res = cross_validate(model, X_train, y_train, cv=cv_strategy,
                             scoring="accuracy", return_train_score=True)
    bias_variance_rows.append({
        "Model": name,
        "Train Accuracy": cv_res["train_score"].mean(),
        "Validation Accuracy": cv_res["test_score"].mean(),
        "Train-Val Gap": cv_res["train_score"].mean() - cv_res["test_score"].mean(),
    })
bias_variance_df = pd.DataFrame(bias_variance_rows).set_index("Model")
bias_variance_df.to_csv(f"{RES_DIR}/bias_variance_analysis.csv")
print("\n=== Bias-Variance Analysis (Train-Validation Gap) ===")
print(bias_variance_df)

fig, ax = plt.subplots(figsize=(9, 5))
x_pos = np.arange(len(bias_variance_df))
width = 0.35
ax.bar(x_pos - width/2, bias_variance_df["Train Accuracy"], width,
       label="Training Accuracy", color="#2980b9")
ax.bar(x_pos + width/2, bias_variance_df["Validation Accuracy"], width,
       label="Validation Accuracy", color="#c0392b")
ax.set_xticks(x_pos)
ax.set_xticklabels(bias_variance_df.index, rotation=15, ha="right")
ax.legend()
_bold_axis_labels(ax, "Model", "Accuracy", "Bias-Variance: Training vs Validation Accuracy")
_save_eps(fig, f"{FIG_DIR}/bias_variance_analysis.eps")
plt.close(fig)

print("\nAll figures saved under:", os.path.abspath(FIG_DIR))
print("All result tables saved under:", os.path.abspath(RES_DIR))
print("\nDone.")

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.



=== Bias-Variance Analysis (Train-Validation Gap) ===
                          Train Accuracy  Validation Accuracy  Train-Val Gap
Model                                                                       
Single Decision Tree            1.000000             0.916484       0.083516
Bagging (Tuned)                 0.991758             0.960440       0.031319
Boosting (Tuned)                1.000000             0.980220       0.019780
Stacked Ensemble (Tuned)        0.964835             0.953846       0.010989

All figures saved under: /home/claude/notebooks/figures
All result tables saved under: /home/claude/notebooks/results

Done.
